# Sampling Preparation

This notebook prepares the MammalWeb MegaDetector dataset for stratified manual sampling.

The sampling is performed separately for:

- humans
- animals
- vehicles

MegaDetector confidence scores are used only to construct confidence strata. They are not treated as ground-truth labels.

The `contains_human` field is retained for reference but is not used as ground truth, for sampling decisions, or for the final performance evaluation.

Planned sampling variables include:

- MegaDetector confidence
- time and lighting condition
- season
- geographic diversity
- bounding-box size
- number of detections
- site diversity
- sequence diversity


In [13]:
import pandas as pd
import numpy as np
from pathlib import Path

In [6]:
df = pd.read_csv("new_mega.csv")

In [5]:
column_names = df.columns.tolist()

print("Number of columns:", len(column_names))

for column in column_names:
    print(column)

Number of columns: 35
photo_id
sequence_id
sequence_num
contains_human
status
filename
dirname
mega_human_confidence
mega_animal_confidence
mega_vehicle_confidence
missing_from_supp
missing_positive_from_supp
site_id
taken
site_name
latitude
longitude
n_human_boxes
n_vehicle_boxes
n_animal_boxes
human_bbox_prob
vehicle_bbox_prob
animal_bbox_prob
human_xmin
vehicle_xmin
animal_xmin
human_ymin
vehicle_ymin
animal_ymin
human_xmax
vehicle_xmax
animal_xmax
human_ymax
vehicle_ymax
animal_ymax


In [6]:
df.shape

(6720367, 35)

In [9]:
metadata_columns = [
    "site_id",
    "taken",
    "site_name",
    "latitude",
    "longitude"
]

metadata_missing = (
    df[metadata_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

metadata_missing

site_id      653334
taken        653334
site_name    653334
latitude     653334
longitude    653334
dtype: int64

In [10]:
metadata_missing_summary = pd.DataFrame({
    "missing_count": df[metadata_columns].isna().sum(),
    "missing_percentage": (
        df[metadata_columns].isna().mean() * 100
    )
}).sort_values("missing_count", ascending=False)

metadata_missing_summary

,missing_count,missing_percentage
site_id,653334,9.721701
taken,653334,9.721701
site_name,653334,9.721701
latitude,653334,9.721701
longitude,653334,9.721701


In [11]:
TARGET_CLASSES = [
    "human",
    "animal",
    "vehicle"
]

SAMPLING_SCORE_COLUMNS = {
    "human": "mega_human_confidence",
    "animal": "mega_animal_confidence",
    "vehicle": "mega_vehicle_confidence"
}

EXCLUDED_REFERENCE_COLUMNS = [
    "contains_human"
]

In [12]:
for target_class, score_column in SAMPLING_SCORE_COLUMNS.items():
    score = df[score_column]

    print(f"\n{target_class.upper()}")
    print("-" * 40)
    print(f"Missing: {(score.isna()).sum():,}")
    print(f"Exactly zero: {(score == 0).sum():,}")
    print(f"Greater than zero: {(score > 0).sum():,}")
    print(f"Maximum: {score.max():.6f}")
    print(f"Mean: {score.mean():.6f}")


HUMAN
----------------------------------------
Missing: 0
Exactly zero: 6,105,807
Greater than zero: 614,560
Maximum: 0.986000
Mean: 0.019873

ANIMAL
----------------------------------------
Missing: 0
Exactly zero: 1,140,706
Greater than zero: 5,579,661
Maximum: 0.993000
Mean: 0.544360

VEHICLE
----------------------------------------
Missing: 0
Exactly zero: 6,040,430
Greater than zero: 679,937
Maximum: 0.987000
Mean: 0.033537


In [13]:
quantile_levels = [
    0,
    0.01,
    0.05,
    0.10,
    0.25,
    0.50,
    0.75,
    0.90,
    0.95,
    0.99,
    1.00
]

for target_class, score_column in SAMPLING_SCORE_COLUMNS.items():
    positive_scores = df.loc[
        df[score_column] > 0,
        score_column
    ]

    print(f"\n{target_class.upper()} positive-confidence quantiles")
    print(positive_scores.quantile(quantile_levels))


HUMAN positive-confidence quantiles
0.00    0.0101
0.01    0.0103
0.05    0.0112
0.10    0.0126
0.25    0.0185
0.50    0.0425
0.75    0.2790
0.90    0.8440
0.95    0.9240
0.99    0.9610
1.00    0.9860
Name: mega_human_confidence, dtype: float64

ANIMAL positive-confidence quantiles
0.00    0.0101
0.01    0.0116
0.05    0.0205
0.10    0.0428
0.25    0.2860
0.50    0.8630
0.75    0.9370
0.90    0.9610
0.95    0.9690
0.99    0.9770
1.00    0.9930
Name: mega_animal_confidence, dtype: float64

VEHICLE positive-confidence quantiles
0.00    0.0101
0.01    0.0105
0.05    0.0124
0.10    0.0155
0.25    0.0340
0.50    0.1900
0.75    0.6270
0.90    0.8660
0.95    0.9210
0.99    0.9600
1.00    0.9870
Name: mega_vehicle_confidence, dtype: float64


## Temporal variable preparation

The image timestamp is converted into a datetime variable so that year,
month, season and hour can later be considered in the sampling design.

Day/night categories are not assigned yet because daylight hours vary by
date and geographic location.

In [14]:
print("taken dtype:", df["taken"].dtype)

df["taken"].dropna().head(10)

taken dtype: object


76     2017-04-09 12:09:18
77     2017-04-09 12:09:20
98     2017-04-09 12:22:35
192    2017-04-09 13:11:03
222    2017-04-09 13:23:58
234    2017-04-09 13:51:20
238    2017-04-09 13:57:29
249    2017-04-09 13:59:55
250    2017-04-09 21:47:49
251    2017-04-09 21:47:51
Name: taken, dtype: object

In [15]:
df["taken_dt"] = pd.to_datetime(
    df["taken"],
    errors="coerce"
)

In [17]:
non_missing_original = df["taken"].notna()
failed_to_parse = non_missing_original & df["taken_dt"].isna()

print(f"Original non-missing timestamps: {non_missing_original.sum():,}")
print(f"Successfully parsed: {df['taken_dt'].notna().sum():,}")
print(f"Failed to parse: {failed_to_parse.sum():,}")

Original non-missing timestamps: 6,067,033
Successfully parsed: 6,066,822
Failed to parse: 211


In [18]:
df.loc[failed_to_parse, "taken"].head(20)

40959      0000-00-00 00:00:00
1520360    0000-05-10 00:00:00
2470767    4337-06-02 04:00:00
2637730    0000-00-00 00:00:00
2954935    7858-11-30 00:00:00
2954936    7859-11-30 00:00:00
2954937    7860-11-30 00:00:00
2954938    7862-11-30 00:00:00
2954939    7866-11-30 00:00:00
2954940    7867-11-30 00:00:00
2954941    7869-11-30 00:00:00
2954942    7870-11-30 00:00:00
2954943    7871-11-30 00:00:00
2954944    7872-11-30 00:00:00
2954945    7873-11-30 00:00:00
2954946    7875-11-30 00:00:00
2954947    7874-11-30 00:00:00
2954948    7876-11-30 00:00:00
2954949    7878-11-30 00:00:00
2954950    7877-11-30 00:00:00
Name: taken, dtype: object

In [20]:
df.loc[
    (df["taken_dt"].dt.year < 2000) |
    (df["taken_dt"].dt.year > 2026),
    "taken_dt"
] = pd.NaT

In [21]:
df["year"] = df["taken_dt"].dt.year
df["month"] = df["taken_dt"].dt.month
df["hour"] = df["taken_dt"].dt.hour
df["date"] = df["taken_dt"].dt.date

season_map = {
    12: "winter",
    1: "winter",
    2: "winter",
    3: "spring",
    4: "spring",
    5: "spring",
    6: "summer",
    7: "summer",
    8: "summer",
    9: "autumn",
    10: "autumn",
    11: "autumn"
}

df["season"] = df["month"].map(season_map)

In [23]:
df[
    ["taken", "taken_dt", "year", "month", "season", "hour"]
].tail(10)

,taken,taken_dt,year,month,season,hour
6720357,2024-09-04 10:30:52,2024-09-04 10:30:52,2024.0,9.0,autumn,10.0
6720358,NaN,NaT,NaN,NaN,NaN,NaN
6720359,NaN,NaT,NaN,NaN,NaN,NaN
6720360,NaN,NaT,NaN,NaN,NaN,NaN
6720361,2024-08-31 17:40:27,2024-08-31 17:40:27,2024.0,8.0,summer,17.0
6720362,2024-08-31 17:40:27,2024-08-31 17:40:27,2024.0,8.0,summer,17.0
6720363,2024-09-04 10:32:39,2024-09-04 10:32:39,2024.0,9.0,autumn,10.0
6720364,2024-08-31 17:40:27,2024-08-31 17:40:27,2024.0,8.0,summer,17.0
6720365,2024-09-04 10:32:39,2024-09-04 10:32:39,2024.0,9.0,autumn,10.0
6720366,2024-09-04 10:32:41,2024-09-04 10:32:41,2024.0,9.0,autumn,10.0


In [24]:
print("Earliest timestamp:", df["taken_dt"].min())
print("Latest timestamp:", df["taken_dt"].max())

Earliest timestamp: 2005-11-30 00:00:00
Latest timestamp: 2026-07-10 10:54:59


In [ ]:
print("Years:")
print(df["year"].value_counts(dropna=False).sort_index())

print("Months:")
print(df["month"].value_counts(dropna=False).sort_index())

print("Seasons:")
print(df["season"].value_counts(dropna=False))

print("Hours:")
print(df["hour"].value_counts(dropna=False).sort_index())


Years:
year
2005.0          1
2012.0        129
2015.0         16
2016.0         60
2017.0       1686
2018.0      20118
2019.0       9772
2020.0     118266
2021.0        386
2022.0       4122
2023.0       7137
2024.0    1940944
2025.0    3707373
2026.0     256788
NaN        653569
Name: count, dtype: int64

Months:
month
1.0      181010
2.0      139588
3.0      130558
4.0      431301
5.0      775656
6.0      774817
7.0      660893
8.0      823205
9.0      697206
10.0    1217843
11.0     198836
12.0      35885
NaN      653569
Name: count, dtype: int64

Seasons:
season
summer    2258915
autumn    2113885
spring    1337515
NaN        653569
winter     356483
Name: count, dtype: int64

Hours:
hour
0.0     134819
1.0     130066
2.0     126687
3.0     118905
4.0     123782
5.0     146126
6.0     169225
7.0     239255
8.0     299244
9.0     331695
10.0    377153
11.0    417945
12.0    437355
13.0    455006
14.0    439414
15.0    399741
16.0    348413
17.0    306940
18.0    241580
19.0    191

## Bounding-box preparation

MegaDetector bounding-box coordinates are stored in pixel coordinates rather
than normalised values.

Bounding-box width, height and raw pixel area are calculated separately for
human, animal and vehicle detections. These values are validated here, but raw
pixel area is not used to define object-size strata because image resolutions
may differ.

For the final sampled images, image width and height will be obtained directly
from the S3 image files. Relative bounding-box area will then be calculated as:

relative bbox area = bbox pixel area / image pixel area

Images without a detection for the relevant class retain missing bounding-box
values. These rows are not removed because they may still be needed for
low-confidence or zero-confidence sampling.

In [ ]:
classes = ["human", "animal", "vehicle"]



HUMAN
Non-missing bbox probability: 614,553
Number of detected boxes: 614,553

ANIMAL
Non-missing bbox probability: 5,579,464
Number of detected boxes: 5,579,464

VEHICLE
Non-missing bbox probability: 679,922
Number of detected boxes: 679,922


In [31]:
for cls in classes:
    coordinate_columns = [
        f"{cls}_xmin",
        f"{cls}_ymin",
        f"{cls}_xmax",
        f"{cls}_ymax"
    ]

    print(f"\n{cls.upper()} coordinate ranges")
    print(df[coordinate_columns].agg(["min", "max"]))


HUMAN coordinate ranges
     human_xmin  human_ymin  human_xmax  human_ymax
min        0.00        0.00     24.8703       3.375
max     6736.61     5117.32   9176.7900    5200.000

ANIMAL coordinate ranges
     animal_xmin  animal_ymin  animal_xmax  animal_ymax
min         0.00         0.00      12.4406      3.37536
max      9004.03      6860.04   13685.1000   8063.19000

VEHICLE coordinate ranges
     vehicle_xmin  vehicle_ymin  vehicle_xmax  vehicle_ymax
min          0.00          0.00       14.2208       4.50048
max       5608.35       5063.76     9067.8000    5200.00000


In [35]:
for cls in classes:
    df[f"{cls}_bbox_width_px"] = (
        df[f"{cls}_xmax"] - df[f"{cls}_xmin"]
    )

    df[f"{cls}_bbox_height_px"] = (
        df[f"{cls}_ymax"] - df[f"{cls}_ymin"]
    )

    df[f"{cls}_bbox_area_px"] = (
        df[f"{cls}_bbox_width_px"]
        * df[f"{cls}_bbox_height_px"]
    )

In [36]:
bbox_validity_summary = []

for cls in classes:
    has_bbox = df[f"{cls}_bbox_prob"].notna()

    missing_coordinate = has_bbox & df[
        [
            f"{cls}_xmin",
            f"{cls}_ymin",
            f"{cls}_xmax",
            f"{cls}_ymax"
        ]
    ].isna().any(axis=1)

    negative_coordinate = has_bbox & (
        (df[f"{cls}_xmin"] < 0) |
        (df[f"{cls}_ymin"] < 0)
    )

    non_positive_size = has_bbox & (
        (df[f"{cls}_bbox_width_px"] <= 0) |
        (df[f"{cls}_bbox_height_px"] <= 0)
    )

    invalid_bbox = (
        missing_coordinate |
        negative_coordinate |
        non_positive_size
    )

    bbox_validity_summary.append({
        "class": cls,
        "has_bbox": int(has_bbox.sum()),
        "missing_coordinate": int(missing_coordinate.sum()),
        "negative_coordinate": int(negative_coordinate.sum()),
        "non_positive_size": int(non_positive_size.sum()),
        "invalid_bbox": int(invalid_bbox.sum())
    })

pd.DataFrame(bbox_validity_summary)

,class,has_bbox,missing_coordinate,negative_coordinate,non_positive_size,invalid_bbox
0,human,614553,0,0,0,0
1,animal,5579464,0,0,0,0
2,vehicle,679922,0,0,0,0


## Geographic preparation

Latitude, longitude and site information are prepared to support geographic
diversity in the manual sample.

`site_id` will not be used as a formal sampling stratum because there are
thousands of sites. Instead, it will be used to limit repeated images from the
same location.

Latitude bands may later be used as broad geographic strata. Their boundaries
will be chosen after inspecting the distribution of unique sites rather than
the distribution of individual photos, so heavily represented sites do not
dominate the boundaries.

In [40]:
df["latitude"] = pd.to_numeric(
    df["latitude"],
    errors="coerce"
)

df["longitude"] = pd.to_numeric(
    df["longitude"],
    errors="coerce"
)

In [41]:
invalid_latitude = (
    df["latitude"].notna()
    & ~df["latitude"].between(-90, 90)
)

invalid_longitude = (
    df["longitude"].notna()
    & ~df["longitude"].between(-180, 180)
)

print(f"Missing site_id: {df['site_id'].isna().sum():,}")
print(f"Missing latitude: {df['latitude'].isna().sum():,}")
print(f"Missing longitude: {df['longitude'].isna().sum():,}")

print(f"Invalid latitude: {invalid_latitude.sum():,}")
print(f"Invalid longitude: {invalid_longitude.sum():,}")

Missing site_id: 653,334
Missing latitude: 653,334
Missing longitude: 653,334
Invalid latitude: 0
Invalid longitude: 0


In [42]:
site_coordinate_check = (
    df.dropna(subset=["site_id"])
    .groupby("site_id")
    .agg(
        latitude_values=("latitude", "nunique"),
        longitude_values=("longitude", "nunique")
    )
)

inconsistent_sites = site_coordinate_check[
    (site_coordinate_check["latitude_values"] > 1)
    | (site_coordinate_check["longitude_values"] > 1)
]

print(f"Unique sites checked: {len(site_coordinate_check):,}")
print(f"Sites with inconsistent coordinates: {len(inconsistent_sites):,}")

inconsistent_sites.head(20)

Unique sites checked: 3,125
Sites with inconsistent coordinates: 0


,latitude_values,longitude_values
site_id,,


In [43]:
site_geo = (
    df.dropna(subset=["site_id"])
    .groupby("site_id", as_index=False)
    .agg(
        site_name=("site_name", "first"),
        latitude=("latitude", "first"),
        longitude=("longitude", "first"),
        n_photos=("photo_id", "size"),
        n_sequences=("sequence_id", "nunique")
    )
)

print("Site-level shape:", site_geo.shape)
print("Sites with latitude:", site_geo["latitude"].notna().sum())
print("Sites with longitude:", site_geo["longitude"].notna().sum())

site_geo.head()

Site-level shape: (3125, 6)
Sites with latitude: 3125
Sites with longitude: 3125


,site_id,site_name,latitude,longitude,n_photos,n_sequences
0,263.0,Edmondsley Wood,54.838001,-1.644982,480,429
1,345.0,Dorset Campsite,50.619343,-1.967860,105,30
2,513.0,Site 2,54.495312,-2.049217,52,8
3,521.0,Site 5,54.499798,-1.813999,828,113
4,523.0,Site 7,54.493202,-1.441007,32,2


In [ ]:
print("Latitude range:")
print(site_geo["latitude"].agg(["min", "max"]))

print("Longitude range:")
print(site_geo["longitude"].agg(["min", "max"]))

Latitude range:
min    -3.881985
max    57.279346
Name: latitude, dtype: float64

Longitude range:
min   -73.206772
max    56.262600
Name: longitude, dtype: float64


In [46]:
geo_quantiles = [
    0,
    0.01,
    0.05,
    0.10,
    0.25,
    0.50,
    0.75,
    0.90,
    0.95,
    0.99,
    1.00
]

print("Site-level latitude quantiles:")
print(site_geo["latitude"].dropna().quantile(geo_quantiles))

print("Site-level longitude quantiles:")
print(site_geo["longitude"].dropna().quantile(geo_quantiles))

Site-level latitude quantiles:
0.00    -3.881985
0.01    41.354427
0.05    50.425928
0.10    50.762633
0.25    51.308071
0.50    52.282154
0.75    54.766293
0.90    56.075683
0.95    57.055836
0.99    57.209924
1.00    57.279346
Name: latitude, dtype: float64
Site-level longitude quantiles:
0.00   -73.206772
0.01    -4.862043
0.05    -4.696544
0.10    -4.155817
0.25    -3.196865
0.50    -1.609183
0.75    -0.959522
0.90    -0.021839
0.95     0.817022
0.99     2.085413
1.00    56.262600
Name: longitude, dtype: float64


In [47]:
site_geo["n_photos"].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
)

count     3125.00000
mean      1941.45056
std       3497.69903
min          1.00000
50%        817.00000
75%       2195.00000
90%       4841.40000
95%       7638.40000
99%      16302.40000
max      70655.00000
Name: n_photos, dtype: float64

In [48]:
site_geo.nlargest(
    20,
    "n_photos"
)[
    [
        "site_id",
        "site_name",
        "latitude",
        "longitude",
        "n_photos",
        "n_sequences"
    ]
]

,site_id,site_name,latitude,longitude,n_photos,n_sequences
1374,6501.0,TR5279_Spotahog_May25,50.272785,-5.180682,70655,5778
471,5441.0,Garden Overall,54.733891,-1.501715,42748,14296
1069,6098.0,Garden Bird feeder,55.195229,-1.648645,40200,13502
223,4958.0,Garden Bird feeder,54.814938,-1.624512,40184,13620
472,5442.0,Garden Bird feeder,54.733849,-1.501616,34496,11620
1011,6019.0,Garden Bird feeder,54.347919,-1.329526,31243,10617
647,5624.0,27_PNL828_QueenElizabethOlympicPark24,51.548473,-0.025386,29436,10921
66,3240.0,Fatfield,54.883991,-1.509279,24103,8357
2585,7781.0,30_NHMP_037_Loddington25,52.614231,-0.824605,23026,6116
759,5740.0,Garden Overall,54.921642,-1.554335,21931,7724


In [49]:
outside_uk_core = (
    ~site_geo["latitude"].between(49, 59)
    | ~site_geo["longitude"].between(-9, 3)
)

print(
    f"Sites outside broad UK coordinate range: "
    f"{outside_uk_core.sum():,}"
)

print(
    f"Photos from those sites: "
    f"{site_geo.loc[outside_uk_core, 'n_photos'].sum():,}"
)

Sites outside broad UK coordinate range: 112
Photos from those sites: 47,773


In [50]:
site_geo.loc[
    outside_uk_core,
    [
        "site_id",
        "site_name",
        "latitude",
        "longitude",
        "n_photos",
        "n_sequences"
    ]
].sort_values("n_photos", ascending=False).head(50)

,site_id,site_name,latitude,longitude,n_photos,n_sequences
2798,8019.0,Rio_pos_1,39.965866,-3.727125,5321,1940
2793,8014.0,Cas_pos_1,39.969883,-3.730137,5319,1994
481,5451.0,"Jabal Yibir, Fujairah",25.647200,56.129700,4917,1666
2799,8020.0,Rio_col_1,39.965851,-3.727050,4219,1530
2797,8018.0,Pin_col_1,39.965946,-3.728379,2670,1041
2813,8049.0,Alcazar_BR526,39.282600,-3.230115,2263,483
284,5137.0,Vinallop,40.779846,0.512023,2053,702
2810,8041.0,BR492,39.000000,-3.000000,1717,329
2796,8017.0,Pin_pos_1,39.965969,-3.728402,1662,650
708,5688.0,MPS_Corriol Castellnou,41.802959,2.104126,1434,446


In [51]:
print(
    "Sites with inconsistent coordinates:",
    len(inconsistent_sites)
)

Sites with inconsistent coordinates: 0


In [52]:
site_geo["geographic_group"] = pd.NA

uk_core = (
    site_geo["latitude"].between(49, 59)
    & site_geo["longitude"].between(-9, 3)
)

site_geo.loc[
    ~uk_core,
    "geographic_group"
] = "outside_uk_core"

In [53]:
site_geo.loc[
    uk_core,
    "geographic_group"
] = pd.qcut(
    site_geo.loc[uk_core, "latitude"],
    q=4,
    labels=[
        "south",
        "south_central",
        "north_central",
        "north"
    ]
)

In [54]:
print(
    site_geo["geographic_group"]
    .value_counts(dropna=False)
)

geographic_group
south              754
north_central      754
south_central      753
north              752
outside_uk_core    112
Name: count, dtype: int64


In [55]:
site_geo.groupby(
    "geographic_group",
    observed=True
)["latitude"].agg(
    site_count="size",
    minimum="min",
    maximum="max",
    median="median"
)

,site_count,minimum,maximum,median
geographic_group,,,,
north,752,54.775791,57.279346,55.876482
north_central,754,52.385143,54.775784,53.202324
outside_uk_core,112,-3.881985,49.348652,41.832150
south,754,50.122993,51.382683,50.866859
south_central,753,51.382706,52.384472,51.539391


In [56]:
site_group_map = site_geo.set_index(
    "site_id"
)["geographic_group"]

df["geographic_group"] = df["site_id"].map(
    site_group_map
)

In [57]:
df["geographic_group"].value_counts(dropna=False)

geographic_group
north_central      2011308
south              1579491
north              1223373
south_central      1205088
NaN                 653334
outside_uk_core      47773
Name: count, dtype: int64

## MegaDetector confidence bands

MegaDetector confidence is used only to stratify the manual sample. It is not
treated as a reference label or evidence that an object is truly present.

Separate confidence-band variables are created for human, animal and vehicle
predictions. Exact zero-confidence images are retained as a distinct category
because they are required for investigating possible false negatives.

The initial band boundaries are exploratory and may be adjusted after examining
the number of available images within each band.

In [58]:
confidence_bins = [
    -0.001,
    0,
    0.02,
    0.05,
    0.10,
    0.30,
    0.70,
    0.90,
    1.001
]

confidence_labels = [
    "zero",
    "0.01_to_0.02",
    "0.02_to_0.05",
    "0.05_to_0.10",
    "0.10_to_0.30",
    "0.30_to_0.70",
    "0.70_to_0.90",
    "0.90_to_1.00"
]

In [59]:
for cls in classes:
    score_col = f"mega_{cls}_confidence"
    band_col = f"{cls}_confidence_band"

    df[band_col] = pd.cut(
        df[score_col],
        bins=confidence_bins,
        labels=confidence_labels,
        include_lowest=True,
        right=True
    )

In [60]:
for cls in classes:
    score_col = f"mega_{cls}_confidence"
    band_col = f"{cls}_confidence_band"

    missing_band = (
        df[score_col].notna()
        & df[band_col].isna()
    )

    print(
        f"{cls}: {missing_band.sum():,} "
        "non-missing scores without a confidence band"
    )

human: 0 non-missing scores without a confidence band
animal: 0 non-missing scores without a confidence band
vehicle: 0 non-missing scores without a confidence band


In [61]:
confidence_band_summaries = {}

for cls in classes:
    band_col = f"{cls}_confidence_band"

    summary = pd.DataFrame({
        "count": df[band_col]
            .value_counts(dropna=False)
            .sort_index()
    })

    summary["percentage"] = (
        summary["count"] / len(df) * 100
    )

    confidence_band_summaries[cls] = summary

    print(f"\n{cls.upper()} confidence bands")
    display(summary)


HUMAN confidence bands


,count,percentage
human_confidence_band,,
zero,6105807,90.855261
0.01_to_0.02,170789,2.541364
0.02_to_0.05,160447,2.387474
0.05_to_0.10,80754,1.201631
0.10_to_0.30,52018,0.774035
0.30_to_0.70,55111,0.820059
0.70_to_0.90,53668,0.798587
0.90_to_1.00,41773,0.621588



ANIMAL confidence bands


,count,percentage
animal_confidence_band,,
zero,1140706,16.973865
0.01_to_0.02,270545,4.025747
0.02_to_0.05,345732,5.144540
0.05_to_0.10,261983,3.898344
0.10_to_0.30,539878,8.033460
0.30_to_0.70,604329,8.992500
0.70_to_0.90,1265191,18.826219
0.90_to_1.00,2292003,34.105325



VEHICLE confidence bands


,count,percentage
vehicle_confidence_band,,
zero,6040430,89.882442
0.01_to_0.02,103721,1.543383
0.02_to_0.05,108307,1.611623
0.05_to_0.10,67915,1.010585
0.10_to_0.30,109045,1.622605
0.30_to_0.70,149710,2.227706
0.70_to_0.90,93246,1.387514
0.90_to_1.00,47993,0.714143


In [62]:
for cls in classes:
    band_col = f"{cls}_confidence_band"

    positive_summary = (
        df.loc[df[f"mega_{cls}_confidence"] > 0, band_col]
        .value_counts()
        .sort_index()
        .to_frame("positive_count")
    )

    positive_summary["positive_percentage"] = (
        positive_summary["positive_count"]
        / positive_summary["positive_count"].sum()
        * 100
    )

    print(f"\n{cls.upper()} positive-confidence distribution")
    display(positive_summary)


HUMAN positive-confidence distribution


,positive_count,positive_percentage
human_confidence_band,,
zero,0,0.000000
0.01_to_0.02,170789,27.790452
0.02_to_0.05,160447,26.107622
0.05_to_0.10,80754,13.140133
0.10_to_0.30,52018,8.464267
0.30_to_0.70,55111,8.967554
0.70_to_0.90,53668,8.732752
0.90_to_1.00,41773,6.797221



ANIMAL positive-confidence distribution


,positive_count,positive_percentage
animal_confidence_band,,
zero,0,0.000000
0.01_to_0.02,270545,4.848771
0.02_to_0.05,345732,6.196290
0.05_to_0.10,261983,4.695321
0.10_to_0.30,539878,9.675821
0.30_to_0.70,604329,10.830927
0.70_to_0.90,1265191,22.675051
0.90_to_1.00,2292003,41.077818



VEHICLE positive-confidence distribution


,positive_count,positive_percentage
vehicle_confidence_band,,
zero,0,0.000000
0.01_to_0.02,103721,15.254502
0.02_to_0.05,108307,15.928976
0.05_to_0.10,67915,9.988425
0.10_to_0.30,109045,16.037515
0.30_to_0.70,149710,22.018216
0.70_to_0.90,93246,13.713918
0.90_to_1.00,47993,7.058448


## Time-of-day preparation

Images are grouped into broad time windows for sampling diversity.

These categories describe clock time rather than exact daylight conditions,
because sunrise and sunset vary by date and location. More precise lighting
information may later be calculated for the sampled images or recorded during
manual annotation.

In [63]:
def assign_time_window(hour):
    if pd.isna(hour):
        return "missing"

    if hour < 5 or hour >= 22:
        return "overnight"

    if hour < 9:
        return "morning_transition"

    if hour < 17:
        return "daytime_core"

    return "evening_transition"


df["time_window"] = df["hour"].apply(assign_time_window)

In [64]:
time_window_order = [
    "overnight",
    "morning_transition",
    "daytime_core",
    "evening_transition",
    "missing"
]

df["time_window"] = pd.Categorical(
    df["time_window"],
    categories=time_window_order,
    ordered=True
)

In [65]:
df["time_window"].value_counts(
    dropna=False
).sort_index()

time_window
overnight              925178
morning_transition     853850
daytime_core          3206722
evening_transition    1081048
missing                653569
Name: count, dtype: int64

In [66]:
time_season_table = pd.crosstab(
    df["season"],
    df["time_window"],
    margins=True
)

time_season_table

time_window,overnight,morning_transition,daytime_core,evening_transition,All
season,,,,,
autumn,359876,305651,1042110,406248,2113885
spring,176969,178732,740333,241481,1337515
summer,319815,314345,1228809,395946,2258915
winter,68518,55122,195470,37373,356483
All,925178,853850,3206722,1081048,6066798


## Sampling-stratum availability

Before drawing the sample, the availability of time, geographic, site and
sequence information is examined within each confidence band.

This is especially important for zero-confidence images because some all-zero
photos were absent from the supplementary metadata export. These missing
records will later be supplemented using a separate metadata query.

In [67]:
availability_summaries = {}

for cls in classes:
    band_col = f"{cls}_confidence_band"

    summary = (
        df.groupby(
            band_col,
            observed=False
        )
        .agg(
            total_images=("photo_id", "size"),
            images_with_time=("taken_dt", lambda x: x.notna().sum()),
            images_with_geography=(
                "geographic_group",
                lambda x: x.notna().sum()
            ),
            unique_sites=("site_id", "nunique"),
            unique_sequences=("sequence_id", "nunique")
        )
    )

    summary["time_coverage_pct"] = (
        summary["images_with_time"]
        / summary["total_images"]
        * 100
    )

    summary["geography_coverage_pct"] = (
        summary["images_with_geography"]
        / summary["total_images"]
        * 100
    )

    availability_summaries[cls] = summary

    print(f"\n{cls.upper()} metadata availability")
    display(summary)


HUMAN metadata availability


,total_images,images_with_time,images_with_geography,unique_sites,unique_sequences,time_coverage_pct,geography_coverage_pct
human_confidence_band,,,,,,,
zero,6105807,5452247,5452480,3120,1495178,89.296091,89.299907
0.01_to_0.02,170789,170788,170789,2095,121095,99.999414,100.000000
0.02_to_0.05,160447,160441,160442,2046,109411,99.996260,99.996884
0.05_to_0.10,80754,80752,80752,1820,61225,99.997523,99.997523
0.10_to_0.30,52018,52018,52018,680,34043,100.000000,100.000000
0.30_to_0.70,55111,55111,55111,699,31711,100.000000,100.000000
0.70_to_0.90,53668,53668,53668,579,27349,100.000000,100.000000
0.90_to_1.00,41773,41773,41773,496,15795,100.000000,100.000000



ANIMAL metadata availability


,total_images,images_with_time,images_with_geography,unique_sites,unique_sequences,time_coverage_pct,geography_coverage_pct
animal_confidence_band,,,,,,,
zero,1140706,487569,487569,2007,420302,42.742740,42.742740
0.01_to_0.02,270545,270538,270538,2789,169472,99.997413,99.997413
0.02_to_0.05,345732,345709,345709,2810,200545,99.993347,99.993347
0.05_to_0.10,261983,261964,261965,2786,165882,99.992748,99.993129
0.10_to_0.30,539878,539814,539833,2953,346947,99.988145,99.991665
0.30_to_0.70,604329,604265,604306,3004,375472,99.989410,99.996194
0.70_to_0.90,1265191,1265112,1265166,3034,612660,99.993756,99.998024
0.90_to_1.00,2292003,2291827,2291947,3033,660845,99.992321,99.997557



VEHICLE metadata availability


,total_images,images_with_time,images_with_geography,unique_sites,unique_sequences,time_coverage_pct,geography_coverage_pct
vehicle_confidence_band,,,,,,,
zero,6040430,5386896,5387111,3123,1475179,89.180671,89.184230
0.01_to_0.02,103721,103714,103716,1266,67315,99.993251,99.995179
0.02_to_0.05,108307,108304,108305,1211,66144,99.997230,99.998153
0.05_to_0.10,67915,67914,67915,1033,44238,99.998528,100.000000
0.10_to_0.30,109045,109035,109044,1003,58872,99.990829,99.999083
0.30_to_0.70,149710,149707,149709,814,63913,99.997996,99.999332
0.70_to_0.90,93246,93236,93241,586,49781,99.989276,99.994638
0.90_to_1.00,47993,47992,47992,432,25452,99.997916,99.997916


In [68]:
confidence_time_tables = {}

for cls in classes:
    band_col = f"{cls}_confidence_band"

    time_available = df.loc[
        df["taken_dt"].notna(),
        [band_col, "time_window"]
    ]

    table = pd.crosstab(
        time_available[band_col],
        time_available["time_window"],
        margins=True,
        dropna=False
    )

    confidence_time_tables[cls] = table

    print(f"\n{cls.upper()}: confidence band × time window")
    display(table)


HUMAN: confidence band × time window


time_window,overnight,morning_transition,daytime_core,evening_transition,missing,All
human_confidence_band,,,,,,
zero,901176,793139,2760795,997137,0,5452247
0.01_to_0.02,8654,17138,123172,21824,0,170788
0.02_to_0.05,7519,16718,115230,20974,0,160441
0.05_to_0.10,3277,9086,57437,10952,0,80752
0.10_to_0.30,829,6266,38857,6066,0,52018
0.30_to_0.70,1114,4772,42021,7204,0,55111
0.70_to_0.90,1823,3961,37517,10367,0,53668
0.90_to_1.00,786,2770,31693,6524,0,41773
All,925178,853850,3206722,1081048,0,6066798



ANIMAL: confidence band × time window


time_window,overnight,morning_transition,daytime_core,evening_transition,missing,All
animal_confidence_band,,,,,,
zero,21118,37142,360445,68864,0,487569
0.01_to_0.02,24617,22307,187478,36136,0,270538
0.02_to_0.05,32202,28125,240085,45297,0,345709
0.05_to_0.10,26405,20863,182420,32276,0,261964
0.10_to_0.30,60542,44780,366315,68177,0,539814
0.30_to_0.70,111470,73603,323188,96004,0,604265
0.70_to_0.90,226112,220750,584568,233682,0,1265112
0.90_to_1.00,422712,406280,962223,500612,0,2291827
All,925178,853850,3206722,1081048,0,6066798



VEHICLE: confidence band × time window


time_window,overnight,morning_transition,daytime_core,evening_transition,missing,All
vehicle_confidence_band,,,,,,
zero,881817,787238,2732410,985431,0,5386896
0.01_to_0.02,8962,10804,70474,13474,0,103714
0.02_to_0.05,8129,11583,74627,13965,0,108304
0.05_to_0.10,4751,7113,47180,8870,0,67914
0.10_to_0.30,7145,10175,77512,14203,0,109035
0.30_to_0.70,8542,11154,109886,20125,0,149707
0.70_to_0.90,4994,9304,62424,16514,0,93236
0.90_to_1.00,838,6479,32209,8466,0,47992
All,925178,853850,3206722,1081048,0,6066798


## Confidence and geographic availability

The number of available images and unique sites is examined across confidence
bands and geographic groups.

Geography is used as a diversity control rather than being fully crossed with
every other sampling variable. Missing geographic metadata is retained as a
separate category during this diagnostic stage.

In [69]:
geo_for_table = (
    df["geographic_group"]
    .astype("object")
    .fillna("missing")
)

confidence_geography_tables = {}

for cls in classes:
    band_col = f"{cls}_confidence_band"

    table = pd.crosstab(
        df[band_col],
        geo_for_table,
        margins=True,
        dropna=False
    )

    confidence_geography_tables[cls] = table

    print(f"\n{cls.upper()}: confidence band × geographic group")
    display(table)


HUMAN: confidence band × geographic group


geographic_group,missing,north,north_central,outside_uk_core,south,south_central,All
human_confidence_band,,,,,,,
zero,653327,1018295,1836831,46906,1455465,1094983,6105807
0.01_to_0.02,0,58449,42490,415,41119,28316,170789
0.02_to_0.05,5,56874,39050,310,36275,27933,160447
0.05_to_0.10,2,29527,21862,133,15609,13621,80754
0.10_to_0.30,0,19547,18978,3,4778,8712,52018
0.30_to_0.70,0,18326,17264,3,6419,13099,55111
0.70_to_0.90,0,12510,19074,1,9587,12496,53668
0.90_to_1.00,0,9845,15759,2,10239,5928,41773
All,653334,1223373,2011308,47773,1579491,1205088,6720367



ANIMAL: confidence band × geographic group


geographic_group,missing,north,north_central,outside_uk_core,south,south_central,All
animal_confidence_band,,,,,,,
zero,653137,139583,128371,144,127477,91994,1140706
0.01_to_0.02,7,55077,81392,504,82093,51472,270545
0.02_to_0.05,23,69584,107265,677,102854,65329,345732
0.05_to_0.10,18,56793,80869,624,75002,48677,261983
0.10_to_0.30,45,122513,160348,1458,150185,105329,539878
0.30_to_0.70,23,122618,207465,3960,152443,117820,604329
0.70_to_0.90,25,268515,458070,14106,282258,242217,1265191
0.90_to_1.00,56,388690,787528,26300,607179,482250,2292003
All,653334,1223373,2011308,47773,1579491,1205088,6720367



VEHICLE: confidence band × geographic group


geographic_group,missing,north,north_central,outside_uk_core,south,south_central,All
vehicle_confidence_band,,,,,,,
zero,653319,1022036,1847477,47397,1392669,1077532,6040430
0.01_to_0.02,5,37124,30735,142,20439,15276,103721
0.02_to_0.05,2,35996,32107,115,21742,18345,108307
0.05_to_0.10,0,21618,18167,60,14670,13400,67915
0.10_to_0.30,1,35917,23855,51,26685,22536,109045
0.30_to_0.70,1,35378,24870,8,64639,24814,149710
0.70_to_0.90,5,27678,18502,0,27342,19719,93246
0.90_to_1.00,1,7626,15595,0,11305,13466,47993
All,653334,1223373,2011308,47773,1579491,1205088,6720367


In [70]:
for cls in classes:
    band_col = f"{cls}_confidence_band"

    temp = df.loc[
        df["site_id"].notna(),
        [band_col, "geographic_group", "site_id"]
    ].copy()

    unique_site_table = (
        temp.groupby(
            [band_col, "geographic_group"],
            observed=True
        )["site_id"]
        .nunique()
        .unstack(fill_value=0)
    )

    print(
        f"\n{cls.upper()}: unique sites by confidence "
        "and geographic group"
    )
    display(unique_site_table)


HUMAN: unique sites by confidence and geographic group


geographic_group,north,north_central,outside_uk_core,south,south_central
human_confidence_band,,,,,
zero,750,753,112,753,752
0.01_to_0.02,510,533,20,523,509
0.02_to_0.05,481,524,15,506,520
0.05_to_0.10,432,474,11,452,451
0.10_to_0.30,179,203,2,116,180
0.30_to_0.70,192,199,3,115,190
0.70_to_0.90,133,172,1,100,173
0.90_to_1.00,123,154,2,81,136



ANIMAL: unique sites by confidence and geographic group


geographic_group,north,north_central,outside_uk_core,south,south_central
animal_confidence_band,,,,,
zero,517,506,17,476,491
0.01_to_0.02,680,690,44,703,672
0.02_to_0.05,687,695,46,704,678
0.05_to_0.10,677,694,49,683,683
0.10_to_0.30,724,718,73,724,714
0.30_to_0.70,730,722,91,728,733
0.70_to_0.90,740,728,105,724,737
0.90_to_1.00,736,732,105,718,742



VEHICLE: unique sites by confidence and geographic group


geographic_group,north,north_central,outside_uk_core,south,south_central
vehicle_confidence_band,,,,,
zero,752,753,112,753,753
0.01_to_0.02,328,341,9,279,309
0.02_to_0.05,308,314,10,274,305
0.05_to_0.10,275,279,7,217,255
0.10_to_0.30,252,268,6,234,243
0.30_to_0.70,171,218,1,222,202
0.70_to_0.90,114,161,0,187,124
0.90_to_1.00,63,125,0,157,87


In [71]:
for cls in classes:
    band_col = f"{cls}_confidence_band"

    temp = df.loc[
        df["sequence_id"].notna(),
        [band_col, "geographic_group", "sequence_id"]
    ].copy()

    unique_sequence_table = (
        temp.groupby(
            [band_col, "geographic_group"],
            observed=True
        )["sequence_id"]
        .nunique()
        .unstack(fill_value=0)
    )

    print(
        f"\n{cls.upper()}: unique sequences by confidence "
        "and geographic group"
    )
    display(unique_sequence_table)


HUMAN: unique sequences by confidence and geographic group


geographic_group,north,north_central,outside_uk_core,south,south_central
human_confidence_band,,,,,
zero,318514,483866,16748,359957,287632
0.01_to_0.02,42944,31887,336,25694,20234
0.02_to_0.05,39194,28593,238,22406,18978
0.05_to_0.10,22019,16998,116,11581,10509
0.10_to_0.30,12133,12239,2,3533,6136
0.30_to_0.70,9843,10306,3,4114,7445
0.70_to_0.90,6422,10270,1,4469,6187
0.90_to_1.00,3924,5839,2,3515,2515



ANIMAL: unique sequences by confidence and geographic group


geographic_group,north,north_central,outside_uk_core,south,south_central
animal_confidence_band,,,,,
zero,63535,53414,112,39056,38162
0.01_to_0.02,38683,51573,449,46078,32683
0.02_to_0.05,45497,62542,592,53820,38084
0.05_to_0.10,38435,52159,551,44114,30613
0.10_to_0.30,77819,104202,1220,94711,68965
0.30_to_0.70,77160,126606,2902,92156,76632
0.70_to_0.90,130375,218579,7996,136091,119602
0.90_to_1.00,133070,224955,10886,161028,130880



VEHICLE: unique sequences by confidence and geographic group


geographic_group,north,north_central,outside_uk_core,south,south_central
vehicle_confidence_band,,,,,
zero,319446,488081,16789,350446,281999
0.01_to_0.02,25030,19738,121,12345,10079
0.02_to_0.05,23272,19331,87,12440,11012
0.05_to_0.10,14990,11997,46,8820,8385
0.10_to_0.30,19480,13564,40,13033,12754
0.30_to_0.70,20174,12322,6,17512,13898
0.70_to_0.90,16611,11176,0,11284,10708
0.90_to_1.00,5226,8802,0,4908,6515


## Sequence diversity preparation

Camera-trap sequences may contain several highly similar images captured within
the same event.

`sequence_id` is used only to prevent repeated or near-duplicate frames from
dominating the manual sample. It is not used to determine whether an object is
truly present and is not treated as a reference label.

The final sampling procedure will normally select no more than one image from
each sequence.

In [72]:
print(f"Missing sequence_id: {df['sequence_id'].isna().sum():,}")
print(f"Unique sequence IDs: {df['sequence_id'].nunique():,}")

Missing sequence_id: 0
Unique sequence IDs: 1,631,937


In [73]:
sequence_sizes = (
    df["sequence_id"]
    .dropna()
    .value_counts(sort=False)
)

sequence_sizes.describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
)

count    1.631937e+06
mean     4.118031e+00
std      4.101103e+00
min      1.000000e+00
50%      3.000000e+00
75%      4.000000e+00
90%      9.000000e+00
95%      1.400000e+01
99%      2.000000e+01
max      2.000000e+01
Name: count, dtype: float64

In [74]:
print(f"Single-photo sequences: {(sequence_sizes == 1).sum():,}")
print(f"Multi-photo sequences: {(sequence_sizes > 1).sum():,}")
print(f"Largest sequence: {sequence_sizes.max():,} photos")

Single-photo sequences: 350,540
Multi-photo sequences: 1,281,397
Largest sequence: 20 photos


In [75]:
sequence_sizes.nlargest(20).to_frame("n_photos")

,n_photos
sequence_id,
27064641,20
27064666,20
27064667,20
27064668,20
27064670,20
27064672,20
27064754,20
27064776,20
27064853,20


In [76]:
sequence_site_counts = (
    df.dropna(subset=["sequence_id", "site_id"])
    .groupby("sequence_id")["site_id"]
    .nunique()
)

print(
    "Sequences linked to more than one site:",
    (sequence_site_counts > 1).sum()
)

Sequences linked to more than one site: 0


In [77]:
df["has_time_metadata"] = df["taken_dt"].notna()

df["has_geographic_metadata"] = (
    df["site_id"].notna()
    & df["latitude"].notna()
    & df["longitude"].notna()
    & df["geographic_group"].notna()
)

df["has_sequence_metadata"] = df["sequence_id"].notna()

In [78]:
df[
    [
        "has_time_metadata",
        "has_geographic_metadata",
        "has_sequence_metadata"
    ]
].mean() * 100

has_time_metadata           90.274802
has_geographic_metadata     90.278299
has_sequence_metadata      100.000000
dtype: float64

In [79]:
for cls in classes:
    df[f"{cls}_has_bbox"] = df[f"{cls}_bbox_prob"].notna()

    df[f"{cls}_positive_missing_bbox"] = (
        (df[f"mega_{cls}_confidence"] > 0)
        & ~df[f"{cls}_has_bbox"]
    )

    print(
        f"{cls}: "
        f"{df[f'{cls}_positive_missing_bbox'].sum():,} "
        "positive-confidence images missing bbox data"
    )

human: 7 positive-confidence images missing bbox data
animal: 197 positive-confidence images missing bbox data
vehicle: 15 positive-confidence images missing bbox data


In [80]:
for cls in classes:
    count_col = f"n_{cls}_boxes"

    print(
        f"{cls}: "
        f"{df[count_col].isna().sum():,} missing counts, "
        f"{(df[count_col] < 0).sum():,} negative counts"
    )

human: 0 missing counts, 0 negative counts
animal: 0 missing counts, 0 negative counts
vehicle: 0 missing counts, 0 negative counts


In [81]:
for cls in classes:
    mismatch = (
        (df[f"n_{cls}_boxes"] > 0)
        != df[f"{cls}_has_bbox"]
    )

    print(f"{cls}: {mismatch.sum():,} count/bbox mismatches")

human: 0 count/bbox mismatches
animal: 0 count/bbox mismatches
vehicle: 0 count/bbox mismatches


In [82]:
population_strata = []

for cls in classes:
    counts = (
        df[f"{cls}_confidence_band"]
        .value_counts(dropna=False)
        .rename_axis("confidence_band")
        .reset_index(name="population_count")
    )

    counts.insert(0, "target_class", cls)
    population_strata.append(counts)

population_strata = pd.concat(
    population_strata,
    ignore_index=True
)

population_strata

,target_class,confidence_band,population_count
0,human,zero,6105807
1,human,0.01_to_0.02,170789
2,human,0.02_to_0.05,160447
3,human,0.05_to_0.10,80754
4,human,0.30_to_0.70,55111
5,human,0.70_to_0.90,53668
6,human,0.10_to_0.30,52018
7,human,0.90_to_1.00,41773
8,animal,0.90_to_1.00,2292003
9,animal,0.70_to_0.90,1265191


In [83]:
population_strata.to_csv(
    "confidence_band_population_counts.csv",
    index=False
)

In [84]:
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]:,}")
print(f"Unique photo IDs: {df['photo_id'].nunique():,}")
print(f"Duplicate photo IDs: {df['photo_id'].duplicated().sum():,}")

assert df["photo_id"].is_unique
assert len(df) == 6_720_367

print("Final preparation checks passed.")

Rows: 6,720,367
Columns: 64
Unique photo IDs: 6,720,367
Duplicate photo IDs: 0
Final preparation checks passed.


In [86]:
df.to_csv(
    "sampling_prepared.csv",
    index=False
)

## Final sampling framework

The manual sample will be constructed separately for:

- Human
- Animal
- Vehicle

Primary stratification:
- MegaDetector confidence

Balancing variables:
- Time window
- Geographic group
- Season

Diversity constraints:
- Maximum one image per sequence
- Maximum one (or two) images per site

Additional variable:
- Relative bounding-box size (to be calculated after candidate sampling)

Ground truth:
- Manual annotation